# Exercise 4 — Clashes and relationships

**Goal:** distinguish declared relationships from geometric overlap. Allow 20–25 minutes. A *clash* here means a detected volumetric intersection; it does not automatically mean a construction error. Run the cells in order.

The bundled wall-and-window sample normally reports zero clashes. We then create two deliberately overlapping walls so you can see a positive result even before your instructor adds a larger example.

In [ ]:
from pathlib import Path
from compas_ifc.bim import BuildingInformationModel

IFC_PATH = Path("sample_building.ifc")  # CHANGE for another public teaching model
if not IFC_PATH.is_file():
    raise FileNotFoundError(f"Missing {IFC_PATH}")
source_model = BuildingInformationModel(str(IFC_PATH))
print("Source elements:", len(list(source_model.elements())))
print("Declared connection edges:", len(source_model.connections))
print("Declared interference edges:", len(source_model.interferences))

## 1. Check the source model

`compute_collisions` compares geometry and records discovered overlaps in the interaction graph. `create_ifc_relations=False` keeps this exercise from writing new relationship entities into the in-memory IFC. The source file on disk is never saved here.

**Predict:** would a wall and its hosted window necessarily count as an unexpected clash?

In [ ]:
found = source_model.compute_collisions(create_ifc_relations=False)
print("New clashes detected:", found)
print("Interference edges now:", len(source_model.interferences))

## 2. Make a known overlap

We use `BuildingInformationModel.template` to make a tiny independent teaching model. `create_wall` adds two boxes to its storey. The second is shifted so part of its volume lies inside the first.

In [ ]:
from compas.geometry import Box, Frame, Point, Vector

demo = BuildingInformationModel.template(schema="IFC4", unit="m")
storey = demo.storeys[0]
for name, xyz in (("Wall A", (0, 0, 0)), ("Wall B", (1, 0.1, 0.2))):
    demo.create_wall(
        name=name,
        parent=storey,
        geometry=Box(4, 0.3, 3),
        frame=Frame(Point(*xyz), Vector.Xaxis(), Vector.Yaxis()),
    )
print("Walls created:", [wall.name for wall in demo.get_elements_by_type("IfcWall")])

## 3. Detect and identify the pair

Predict the count before running. The graph stores each detected pair as an edge; `node_element` gets the IFC object at each end.

In [ ]:
new_clashes = demo.compute_collisions(element_types=["IfcWall"], create_ifc_relations=False)
print("New clashes:", new_clashes)
for a, b in demo.interferences:
    first = demo.graph.node_element(a)
    second = demo.graph.node_element(b)
    print(first.name, "overlaps", second.name)

## 4. See the teaching model in Binder

The notebook Viewer shows the two meshes. Look for their overlap; the computed clash count remains the evidence from `compute_collisions`.

In [ ]:
from compas_notebook.viewer import Viewer

viewer = Viewer()
for wall in demo.get_elements_by_type("IfcWall"):
    mesh = wall.compute_surface_mesh().transformed(wall.compute_modeltransformation())
    viewer.scene.add(mesh)
viewer.show()

## Your turn

- Move Wall B further away by changing `(1, 0.1, 0.2)` to `(10, 0.1, 0.2)`. Rerun from **Make a known overlap**. What changes?
- Restore the overlap. Why is one detected pair not enough to decide whether a real design is acceptable?
- When your instructor supplies a larger IFC, change `IFC_PATH` and repeat the source scan. It may take longer. Record the model's units and which element classes were considered.

[Upstream interactions tutorial](https://compas.dev/compas_ifc/latest/tutorials/04_interactions.html).